Take a sample string of 16 bytes that is to be Encrypted using AES 128

In [1]:
text_inp = "This is a sample"

Convert given string into standard bytes

In [2]:
byte_data = text_inp.encode('utf-8')

Convert into a list

In [3]:
byte_list = list(byte_data)
print(byte_list) #check if all values are in ascii

[84, 104, 105, 115, 32, 105, 115, 32, 97, 32, 115, 97, 109, 112, 108, 101]


Define functions that can convert bytes to state(required format for AES) and vice versa

In [4]:
def bytes_to_state(byte_list):      #converts bytes to state
    # Initialize an empty 4x4 matrix
    state = [[0] * 4 for _ in range(4)]
    
    # Fill the matrix column by column
    for r in range(4):
        for c in range(4):
            state[r][c] = byte_list[r + 4 * c]
            
    return state

def state_to_bytes(state):          # converts state to bytes
    flat_list = []
    for c in range(4):
        for r in range(4):
            flat_list.append(state[r][c])
    return flat_list

Now use those functions and check if they work before proceeding. To check, just use the first function bytes_to_state to convert the current byte_list into a state format, then use that state with the second function(state_to_bytes) to check if you can get back the original byte_list.

In [5]:
state_test = bytes_to_state(byte_list)         
bytes_test = state_to_bytes(state_test)

if(bytes_test == byte_list):                    # checks if both functions work as expected
    print("Functions Verification Success")
else:
    print("Functions Verification Failed")

Functions Verification Success


Once we see Functions Verification Success, we can proceed to use these two functions for our code according to our requirements.

Now we use the function to get the state format we want after verification.

In [6]:
state = bytes_to_state(byte_list)
print(state)                                   

[[84, 32, 97, 109], [104, 105, 32, 112], [105, 115, 115, 108], [115, 32, 97, 101]]


Now, we define the S-Box for the SubBytes transformation

In [7]:
S_Box = [
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16
]

Define the SubBytes function using the S_Box for substitution

In [8]:
def sub_bytes(state):
    for r in range(4):
        for c in range(4):
            byte_val = state[r][c]
            state[r][c] = S_Box[byte_val]
    pass

Define the ShiftRows function that shifts left using Python Slicing

In [9]:
def shift_rows(state):
    for r in range(4):
        state[r] = state[r][r:] + state[r][:r]
    pass

Define the xtime(b) function to be used by the MixColumns function

In [10]:
def xtime(b):
    shifted = (b << 1) & 0xFF
    if(b & 0x80 != 0):
        return shifted ^ 0x1B
    else:
        return shifted
    pass

Define the MixColumns function using the xtime(b) function

In [11]:
def mul_2(b): return xtime(b)
def mul_3(b): return xtime(b) ^ b

def mix_columns(state):
    for c in range(4):
        s0 = state[0][c] 
        s1 = state[1][c] 
        s2 = state[2][c] 
        s3 = state[3][c]

        state[0][c] = mul_2(s0) ^ mul_3(s1) ^ s2        ^ s3
        state[1][c] = s0        ^ mul_2(s1) ^ mul_3(s2) ^ s3
        state[2][c] = s0        ^ s1        ^ mul_2(s2) ^ mul_3(s3)
        state[3][c] = mul_3(s0) ^ s1        ^ s2        ^ mul_2(s3)

Define the AddRoundKey function

In [12]:
def add_round_key(state, round_key):
    for r in range(4):
        for c in range(4):
            state[r][c] = state[r][c] ^ round_key[r][c]

Define the Round Constants for the KeyExpansion algorithm (as described in Table 5)

In [13]:
Rcon = [
    [0x00, 0x00, 0x00, 0x00], # Index 0 is unused as we need 10 Round Constants
    [0x01, 0x00, 0x00, 0x00], [0x02, 0x00, 0x00, 0x00],
    [0x04, 0x00, 0x00, 0x00], [0x08, 0x00, 0x00, 0x00],
    [0x10, 0x00, 0x00, 0x00], [0x20, 0x00, 0x00, 0x00],
    [0x40, 0x00, 0x00, 0x00], [0x80, 0x00, 0x00, 0x00],
    [0x1b, 0x00, 0x00, 0x00], [0x36, 0x00, 0x00, 0x00]
]

Define the RotWord Function for the KeyExpansion algorithm

In [14]:
def rot_word(word):
    return word[1:] + word[:1]

Define the SubWord Function for the KeyExpansion algorithm

In [15]:
def sub_word(word):
    return [S_Box[i] for i in word]

Define the Key Expansion algorithm using the functions created above

In [16]:
def key_expansion(key):
    w = []

    for i in range(4):
        word = key[4*i : 4*i + 4]
        w.append(word)

    for i in range(4, 44):
        temp = w[i - 1][:]
        
        if i % 4 == 0:
            temp = sub_word(rot_word(temp))
            rcon = Rcon[i // 4]
            temp = [temp[b] ^ rcon[b] for b in range(4)]

        new_word = [w[i - 4][b] ^ temp[b] for b in range(4)]
        w.append(new_word)
        
    return w

Now, use the functions defined above to build the Main Encryption Function.

Before proceeding, the AddRoundKey function expects a 4x4 matrix so we build a helper function to create the required matrix

In [17]:
def get_round_key_matrix(w, round_num):

    round_words = w[4 * round_num : 4 * round_num + 4]
    
    flat_bytes = []
    for word in round_words:
        flat_bytes.extend(word)

    return bytes_to_state(flat_bytes)

Define the Main Encryption Function

In [18]:
def aes_128_encrypt(input, key):

    state = bytes_to_state(input)
    w = key_expansion(key)
    round_0_key = get_round_key_matrix(w, 0)
    add_round_key(state, round_0_key)
    
    # Rounds 1 to 9
    for r in range(1, 10):
        sub_bytes(state)
        shift_rows(state)
        mix_columns(state)
        round_key = get_round_key_matrix(w, r)
        add_round_key(state, round_key)
        
    # Final Round
    sub_bytes(state)
    shift_rows(state)
    round_10_key = get_round_key_matrix(w, 10)
    add_round_key(state, round_10_key)

    return state_to_bytes(state)

Test the Main Encryption Function for the text_inp. 
This helps us test using a sample input and key of our choice where both the input and key are strings. 
However, here we do not know for sure what the expected output is, so we compare the length of the ciphertext with expected length.

In [19]:
def test_full_aes_128():
    key_string = "SecretKey2607004"
    key = list(key_string.encode('utf-8'))

    ciphertext = aes_128_encrypt(byte_list, key)

    if len(ciphertext) == 16:
        print("AES-128 Encryption Test Successful (For Sample Text Input)")
    else:
        print("AES-128 Encryption Test Failed (For Sample Text Input)")

test_full_aes_128()

AES-128 Encryption Test Successful (For Sample Text Input)


Test the Main Encryption Function for Appendix B. This is the test included in Appendix B of the FIPS 197 document for AES. This has a sample input , key and expected output which gives us the best method to test our function to decode errors.

In [20]:
def test_full_aes_128(): # Test from Appendix B

    input = [
        0x32, 0x43, 0xf6, 0xa8, 0x88, 0x5a, 0x30, 0x8d,
        0x31, 0x31, 0x98, 0xa2, 0xe0, 0x37, 0x07, 0x34
    ]

    key = [
        0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6,
        0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c
    ]
    
    expected_output = [
        0x39, 0x25, 0x84, 0x1d, 0x02, 0xdc, 0x09, 0xfb,
        0xdc, 0x11, 0x85, 0x97, 0x19, 0x6a, 0x0b, 0x32
    ]

    actual_output = aes_128_encrypt(input, key)

    if(actual_output == expected_output):
        print("AES-128 Encryption Test Successful (Appendix B Test)")
    else:
        print("AES-128 Encryption Test Failed (Appendix B Test)")

test_full_aes_128()

AES-128 Encryption Test Successful (Appendix B Test)


Now, lets write the Decryption algorithm.
We will need to reverse the steps we did in order to reach to this point.

First we will need an Inverse S-Box.

In [21]:
Inv_S_Box = [
    0x52, 0x09, 0x6a, 0xd5, 0x30, 0x36, 0xa5, 0x38, 0xbf, 0x40, 0xa3, 0x9e, 0x81, 0xf3, 0xd7, 0xfb,
    0x7c, 0xe3, 0x39, 0x82, 0x9b, 0x2f, 0xff, 0x87, 0x34, 0x8e, 0x43, 0x44, 0xc4, 0xde, 0xe9, 0xcb,
    0x54, 0x7b, 0x94, 0x32, 0xa6, 0xc2, 0x23, 0x3d, 0xee, 0x4c, 0x95, 0x0b, 0x42, 0xfa, 0xc3, 0x4e,
    0x08, 0x2e, 0xa1, 0x66, 0x28, 0xd9, 0x24, 0xb2, 0x76, 0x5b, 0xa2, 0x49, 0x6d, 0x8b, 0xd1, 0x25,
    0x72, 0xf8, 0xf6, 0x64, 0x86, 0x68, 0x98, 0x16, 0xd4, 0xa4, 0x5c, 0xcc, 0x5d, 0x65, 0xb6, 0x92,
    0x6c, 0x70, 0x48, 0x50, 0xfd, 0xed, 0xb9, 0xda, 0x5e, 0x15, 0x46, 0x57, 0xa7, 0x8d, 0x9d, 0x84,
    0x90, 0xd8, 0xab, 0x00, 0x8c, 0xbc, 0xd3, 0x0a, 0xf7, 0xe4, 0x58, 0x05, 0xb8, 0xb3, 0x45, 0x06,
    0xd0, 0x2c, 0x1e, 0x8f, 0xca, 0x3f, 0x0f, 0x02, 0xc1, 0xaf, 0xbd, 0x03, 0x01, 0x13, 0x8a, 0x6b,
    0x3a, 0x91, 0x11, 0x41, 0x4f, 0x67, 0xdc, 0xea, 0x97, 0xf2, 0xcf, 0xce, 0xf0, 0xb4, 0xe6, 0x73,
    0x96, 0xac, 0x74, 0x22, 0xe7, 0xad, 0x35, 0x85, 0xe2, 0xf9, 0x37, 0xe8, 0x1c, 0x75, 0xdf, 0x6e,
    0x47, 0xf1, 0x1a, 0x71, 0x1d, 0x29, 0xc5, 0x89, 0x6f, 0xb7, 0x62, 0x0e, 0xaa, 0x18, 0xbe, 0x1b,
    0xfc, 0x56, 0x3e, 0x4b, 0xc6, 0xd2, 0x79, 0x20, 0x9a, 0xdb, 0xc0, 0xfe, 0x78, 0xcd, 0x5a, 0xf4,
    0x1f, 0xdd, 0xa8, 0x33, 0x88, 0x07, 0xc7, 0x31, 0xb1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xec, 0x5f,
    0x60, 0x51, 0x7f, 0xa9, 0x19, 0xb5, 0x4a, 0x0d, 0x2d, 0xe5, 0x7a, 0x9f, 0x93, 0xc9, 0x9c, 0xef,
    0xa0, 0xe0, 0x3b, 0x4d, 0xae, 0x2a, 0xf5, 0xb0, 0xc8, 0xeb, 0xbb, 0x3c, 0x83, 0x53, 0x99, 0x61,
    0x17, 0x2b, 0x04, 0x7e, 0xba, 0x77, 0xd6, 0x26, 0xe1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0c, 0x7d
]

Now we need to define the inverse version of the functions we used one by one.

We can start by defining the inverse SubBytes function(Using the Inverse S-Box).

In [22]:
def inv_sub_bytes(state):
    for r in range(4):
        for c in range(4):
            state[r][c] = Inv_S_Box[state[r][c]]

Next we need to define the inverse ShiftRows function.

In [23]:
def inv_shift_rows(state):
    for r in range(4):
        state[r] = state[r][-r:] + state[r][:-r]

Now lets define the Inverse Mix Columns function. For defining the Mix Columns function we did matrix multiplication of the the state matrix with a particular matrix. In order to inverse the Mix columns function, we just need to multiply the inverse of the original matrix with the state matrix. Therefore we define four helper functions for four different values in the inverse matrix. Each value is broken down to a binary expression(all in powers of 2) and converted using the xtime function. For example 14 is broken down as 2^3 + 2^2 + 2^1.

In [24]:
def mul_9(b):  return xtime(xtime(xtime(b))) ^ b
def mul_11(b): return xtime(xtime(xtime(b)) ^ b) ^ b
def mul_13(b): return xtime(xtime(xtime(b) ^ b)) ^ b
def mul_14(b): return xtime(xtime(xtime(b) ^ b) ^ b)

def inv_mix_columns(state):
    for c in range(4):
        s0, s1, s2, s3 = state[0][c], state[1][c], state[2][c], state[3][c]

        state[0][c] = mul_14(s0) ^ mul_11(s1) ^ mul_13(s2) ^ mul_9(s3)
        state[1][c] = mul_9(s0)  ^ mul_14(s1) ^ mul_11(s2) ^ mul_13(s3)
        state[2][c] = mul_13(s0) ^ mul_9(s1)  ^ mul_14(s2) ^ mul_11(s3)
        state[3][c] = mul_11(s0) ^ mul_13(s1) ^ mul_9(s2)  ^ mul_14(s3)

We do not need a reverse function for AddRoundKey since Xor is its own reverse.

Now that we have all the functions we need, we can now define the Decryption function.
We start with where we ended in the Encryption function ie Round 10 and then we go from 9 to 1 (in reverse order).

In [25]:
def aes_128_decrypt(ciphertext_bytes, key_bytes):
    state = bytes_to_state(ciphertext_bytes)
    w = key_expansion(key_bytes)

    # First Round in Decryption = Last round in Encryption
    add_round_key(state, get_round_key_matrix(w, 10))
    
    # Rounds 9 to 1
    for r in range(9, 0, -1):
        inv_shift_rows(state)
        inv_sub_bytes(state)
        add_round_key(state, get_round_key_matrix(w, r))
        inv_mix_columns(state)
        
    # Final Round (Round 0)
    inv_shift_rows(state)
    inv_sub_bytes(state)
    add_round_key(state, get_round_key_matrix(w, 0))
    
    return state_to_bytes(state)

Now Lets test the Decryption Function using the sample text we used first, this time if we get the Decrypted Function the same as the original function , that would mean both Encryption and Decryption functions work correctly.

In [26]:
def test_decrypt_encrypt_aes_128():
    key_string = "SecretKey2607004"
    key = list(key_string.encode('utf-8'))

    ciphertext = aes_128_encrypt(byte_list, key)

    decryptedtext = aes_128_decrypt(ciphertext,key)

    if byte_list == decryptedtext:
        print("AES-128 Encryption and Decryption Test Successful (For Sample Text Input)")
    else:
        print("AES-128 Encryption and Decryption Test Failed (For Sample Text Input)")

test_decrypt_encrypt_aes_128()

AES-128 Encryption and Decryption Test Successful (For Sample Text Input)


Finally, lets test the Decryption Function using the Input and Key given in Appendix B.

In [27]:
def test_decrypt_encrypt_aes_128():
    input = [
            0x32, 0x43, 0xf6, 0xa8, 0x88, 0x5a, 0x30, 0x8d,
            0x31, 0x31, 0x98, 0xa2, 0xe0, 0x37, 0x07, 0x34
        ]
    
    key = [
            0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6,
            0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c
        ]

    encrypted_text = aes_128_encrypt(input,key)

    decrypted_text = aes_128_decrypt(encrypted_text,key)

    if(decrypted_text == input):
        print("AES-128 Encryption and Decryption Test Successful (Appendix B Test)")
    else:
        print("AES-128 Encryption and Decryption Test Failed (Appendix B Test)")

test_decrypt_encrypt_aes_128()

AES-128 Encryption and Decryption Test Successful (Appendix B Test)


Now we create a custom test using custom a custom key and sample but this time we also display the actual decrypted text and input text to visually prove that the function works as expected.

In [28]:
def sample_test():
    sample_txt = "SampleText123456"
    input = list(sample_txt.encode('utf-8'))

    key_txt = "MyPassword123456"
    key = list(key_txt.encode('utf-8'))

    encrypted_text = aes_128_encrypt(input,key)

    decrypted_text = aes_128_decrypt(encrypted_text,key)

    if(decrypted_text == input):
        print("AES-128 Encryption and Decryption Test Successful (Custom Test)")
        print(f'Given Input = {bytes(input).decode('utf-8')}')
        print(f'Decrypted Text = {bytes(decrypted_text).decode('utf-8')}')
    else:
        print("AES-128 Encryption and Decryption Test Failed (Custom Test)")
        print(f'Given Input = {bytes(input).decode('utf-8')}')
        print(f'Decrypted Text = {bytes(decrypted_text).decode('utf-8')}')

sample_test()

AES-128 Encryption and Decryption Test Successful (Custom Test)
Given Input = SampleText123456
Decrypted Text = SampleText123456
